# 14 - ML 模型训练（XGBoost 多 horizon 逐票）

对每只标的，独立训练 3 个 XGBoost 回归模型，分别预测 3 / 5 / 10 日前瞻对数收益。

**前置依赖：**
- `stock_signal_snapshot` 表必须有充足历史（用 `CompositeScorer.backfill_history` 回填）
- `stock_composite_score` 表用于构造 composite 时序特征
- `stock_daily / stock_technical / stock_fundamental` 必须覆盖训练区间

**步骤：**
1. 初始化 + 检查数据可用性
2. 历史数据回填（首次/扩容时跑）
3. 逐票构造特征矩阵 + 标签
4. 走 walk-forward CV 训练
5. 落库 + 输出评估报告

## 1. 初始化

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from datetime import datetime

from invest_model.db import get_engine
from invest_model.models.ddl import create_all_tables
from invest_model.repositories.stock_pool_repo import StockPoolRepository
from invest_model.repositories.calendar_repo import CalendarRepository

engine = get_engine()
create_all_tables(engine)  # 确保 ml_model_registry 等表已建

pool_repo = StockPoolRepository(engine)
_core = pool_repo.get_pool('core')
_etf = pool_repo.get_pool('etf')
all_pool = pd.concat([_core, _etf], ignore_index=True)
all_codes = all_pool['code'].tolist()
code_name_map = dict(zip(all_pool['code'], all_pool['name']))

TRAIN_START = '20210101'   # 训练数据起点（5 年回看）
OOS_CUTOFF  = '20241231'   # OOS 切分点：之前用于 v1_oos 训练，之后留作 15_ml_backtest 严格 OOS 回测
TRAIN_END   = datetime.now().strftime('%Y%m%d')

cal = CalendarRepository(engine)
all_trade_dates = cal.get_trade_dates(TRAIN_START, TRAIN_END)
if all_trade_dates:
    TRAIN_END = all_trade_dates[-1]

oos_dates = cal.get_trade_dates(TRAIN_START, OOS_CUTOFF)
n_oos_train = len(oos_dates) if oos_dates else 0
n_oos_test = len(all_trade_dates) - n_oos_train

print(f'训练区间: {TRAIN_START} ~ {TRAIN_END} ({len(all_trade_dates)} 个交易日)')
print(f'OOS 切分点: {OOS_CUTOFF}')
print(f'  → v1     (live):     训练数据 {TRAIN_START} ~ {TRAIN_END}    [{len(all_trade_dates)} 天]   供 13_daily_advisor 实盘')
print(f'  → v1_oos (backtest): 训练数据 {TRAIN_START} ~ {OOS_CUTOFF}    [{n_oos_train} 天]   供 15_ml_backtest 严格 OOS 回测')
print(f'    (回测期 ≈ {OOS_CUTOFF} ~ {TRAIN_END}, 约 {n_oos_test} 个 OOS 交易日)')
print(f'标的池: {len(all_codes)} 只 (core {len(_core)} + etf {len(_etf)})')
for c in all_codes:
    print(f'  {c} {code_name_map.get(c, "")}')

训练区间: 20210101 ~ 20260514 (1296 个交易日)
OOS 切分点: 20241231
  → v1     (live):     训练数据 20210101 ~ 20260514    [1296 天]   供 13_daily_advisor 实盘
  → v1_oos (backtest): 训练数据 20210101 ~ 20241231    [969 天]   供 15_ml_backtest 严格 OOS 回测
    (回测期 ≈ 20241231 ~ 20260514, 约 327 个 OOS 交易日)
标的池: 7 只 (core 6 + etf 1)
  000833.SZ 粤桂股份
  002156.SZ 通富微电
  002594.SZ 比亚迪
  002648.SZ 卫星化学
  300442.SZ 润泽科技
  600118.SH 中国卫星
  516120.SH 化工50ETF


## 2. 历史数据回填（信号快照 + 综合分）

首次训练或扩容历史区间时运行。增量执行，已有数据自动跳过。

**注意：** 这一步可能耗时较长（逐日跑全部 generator）。

> **⚠ TRAIN_START 已从 2022-01-01 拉长到 2021-01-01（5 年）**
> 如果之前 `stock_signal_snapshot` / `stock_composite_score` 的数据仅覆盖到 2022-01-01，需要把 `RUN_BACKFILL = True`，回填 2021 年那段历史。`backfill_history` 会自动跳过已有日期，只补缺口。

In [2]:
from invest_model.scoring.scorer import CompositeScorer

RUN_BACKFILL = True  # 首次设 True；后续日常增量交给 daily_pipeline

if RUN_BACKFILL:
    scorer = CompositeScorer(engine)
    n = scorer.backfill_history(
        codes=all_codes,
        start_date=TRAIN_START,
        end_date=TRAIN_END,
        step_days=1,
        skip_existing=True,
        persist=True,
    )
    print(f'回填完成: 综合分 {n} 条')
else:
    print('跳过回填（设 RUN_BACKFILL=True 启用）')

17:43:02 | INFO    | backfill: 区间 20210101~20260514 共 1296 个交易日, 本次实际打分 1296 天, 7 只股票
17:43:10 | INFO    | backfill 进度: 10/1296
17:43:19 | INFO    | backfill 进度: 20/1296
17:43:28 | INFO    | backfill 进度: 30/1296
17:43:36 | INFO    | backfill 进度: 40/1296
17:43:45 | INFO    | backfill 进度: 50/1296
17:43:54 | INFO    | backfill 进度: 60/1296
17:44:07 | INFO    | backfill 进度: 70/1296
17:44:22 | INFO    | backfill 进度: 80/1296
17:44:36 | INFO    | backfill 进度: 90/1296
17:44:51 | INFO    | backfill 进度: 100/1296
17:45:06 | INFO    | backfill 进度: 110/1296
17:45:21 | INFO    | backfill 进度: 120/1296
17:45:36 | INFO    | backfill 进度: 130/1296
17:45:51 | INFO    | backfill 进度: 140/1296
17:46:07 | INFO    | backfill 进度: 150/1296
17:46:22 | INFO    | backfill 进度: 160/1296
17:46:37 | INFO    | backfill 进度: 170/1296
17:46:52 | INFO    | backfill 进度: 180/1296
17:47:08 | INFO    | backfill 进度: 190/1296
17:47:24 | INFO    | backfill 进度: 200/1296
17:47:39 | INFO    | backfill 进度: 210/1296
17:47:55 | INFO    |

回填完成: 综合分 9072 条


## 3. 构造特征矩阵 + 标签

对每只标的，调用 `FeatureBuilder.build_history` 生成完整特征矩阵；
调用 `make_forward_returns` 生成 3 / 5 / 10 日前瞻对数收益作为多 horizon 标签。

In [3]:
from invest_model.ml import FeatureBuilder, make_forward_returns, LABEL_HORIZONS
from invest_model.repositories.stock_daily_repo import StockDailyRepository
from invest_model.repositories.etf_repo import ETFRepository

builder = FeatureBuilder(engine)
daily_repo = StockDailyRepository(engine)
etf_repo = ETFRepository(engine)

stock_data: dict[str, dict] = {}
for code in all_codes:
    X = builder.build_history(code, TRAIN_START, TRAIN_END)
    if X.empty:
        print(f'  [skip] {code}: 特征矩阵为空')
        continue

    daily = daily_repo.get_daily(code, TRAIN_START, TRAIN_END)
    if daily.empty:
        daily = etf_repo.get_daily(code, TRAIN_START, TRAIN_END)
    if daily.empty:
        print(f'  [skip] {code}: 日线为空')
        continue

    labels = make_forward_returns(daily, horizons=LABEL_HORIZONS)
    labels = labels.set_index('trade_date')

    # 对齐索引 + 转每个 horizon 一个 Series
    aligned = X.join(labels, how='left')
    y_dict = {h: aligned[f'fwd_ret_{h}d'] for h in LABEL_HORIZONS}
    X_clean = aligned[list(X.columns)]

    stock_data[code] = {'X': X_clean, 'y': y_dict, 'name': code_name_map.get(code, '')}
    n_valid_5d = aligned['fwd_ret_5d'].notna().sum()
    print(
        f'  {code} {code_name_map.get(code, ""):8s} | features={X_clean.shape} | '
        f'valid_5d={n_valid_5d}'
    )

  000833.SZ 粤桂股份     | features=(1232, 60) | valid_5d=1225
  002156.SZ 通富微电     | features=(1296, 60) | valid_5d=1289
  002594.SZ 比亚迪      | features=(1232, 60) | valid_5d=1225
  002648.SZ 卫星化学     | features=(1232, 60) | valid_5d=1225
  300442.SZ 润泽科技     | features=(1232, 60) | valid_5d=1219
  600118.SH 中国卫星     | features=(1296, 60) | valid_5d=1289
  516120.SH 化工50ETF  | features=(1255, 60) | valid_5d=1248


## 4. 逐票训练 walk-forward XGBoost

每只标的对 3 / 5 / 10 三个 horizon 分别训练，输出 IC、RMSE、命中率。

In [4]:
from invest_model.ml import (
    PerStockMultiHorizonTrainer,
    PurgedWalkForwardSplit,
    save_results,
    effective_feature_columns,
)

MODEL_VERSION_LIVE = 'v1'        # 全量训练 → 用于 13_daily_advisor 实盘推理
MODEL_VERSION_OOS  = 'v1_oos'    # 截断到 OOS_CUTOFF → 用于 15_ml_backtest 严格 OOS 回测

# ETF 池 → ETF 训练时走精简列（19~20 维，含 RSI/MACD/波动率等纯价格衍生）；
# 同时 trainer 内部会对 ETF 自动套用 ETF_PARAMS_OVERLAY（更激进的分裂参数），
# 避免低标签波动 + 强正则 → 输出退化为均值 → IC=0 的死局。
_etf_codes_set = set(_etf['code'].tolist()) if not _etf.empty else set()

# v1 (live)：5 年全量 ≈ 1227 行样本，train_window=1000（约 4 年）有充足历史
cv_live = PurgedWalkForwardSplit(
    n_splits=5, train_window=1000, val_window=60, embargo=10,
)
trainer_live = PerStockMultiHorizonTrainer(cv=cv_live, etf_codes=_etf_codes_set)

# v1_oos：截到 2024-12-31 后 dropna 仅约 900-970 行有效样本，必须把 train_window 调到 600 才能 fit
#   阈值 = train_window + val_window + embargo = 600 + 60 + 10 = 670 < 900 ✓
#   5 折滑动：第 4 折需 train_window + val_window + embargo + (n_splits-1)*val_window = 910 ≈ 数据量
cv_oos = PurgedWalkForwardSplit(
    n_splits=5, train_window=600, val_window=60, embargo=10,
)
trainer_oos = PerStockMultiHorizonTrainer(cv=cv_oos, etf_codes=_etf_codes_set)


def _train_and_save(trainer, version: str, x_filter_end: str | None) -> int:
    """统一训练流程：可选地把每只票的特征矩阵截断到 x_filter_end。"""
    print('=' * 70)
    tw = trainer.cv.train_window
    if x_filter_end:
        print(f'  开始训练 [{version}]   截断到 {x_filter_end}（OOS 模型）  cv.train_window={tw}')
    else:
        print(f'  开始训练 [{version}]   全量训练（live 模型）  cv.train_window={tw}')
    print('=' * 70)

    results: list = []
    for code, data in stock_data.items():
        if x_filter_end:
            X_use = data['X'][data['X'].index <= x_filter_end]
            y_use = {h: y[y.index <= x_filter_end] for h, y in data['y'].items()}
        else:
            X_use = data['X']
            y_use = data['y']

        # ETF 走精简列；其他股票用完整列
        eff_cols = effective_feature_columns(code, etf_codes=_etf_codes_set)
        eff_cols = [c for c in eff_cols if c in X_use.columns]
        X_use = X_use[eff_cols]

        if X_use.empty:
            print(f'  [skip] {code}: 截断后无样本')
            continue
        n_valid = (~y_use[5].isna()).sum() if 5 in y_use else len(X_use)
        is_etf = code in _etf_codes_set
        tag = 'ETF精简' if is_etf else '完整'
        print(f'\n=== {code} {data["name"]:8s}  样本 {len(X_use)} | y_5d_valid {n_valid} | 特征={X_use.shape[1]}维({tag}) ===')
        res = trainer.fit(code, X_use, y_use)
        if res.horizons:
            results.append(res)

    n_saved = save_results(engine, results, version=version)
    print(f'\n→ 落库 {n_saved} 个 (code, horizon) 模型 [version={version}]')
    return n_saved


# 1. 先训 v1_oos（截断到 OOS_CUTOFF），结果干净可用于 15_ml_backtest 真 OOS 回测
n_oos = _train_and_save(trainer_oos, MODEL_VERSION_OOS, x_filter_end=OOS_CUTOFF)

# 2. 再训 v1（全量），用于 13_daily_advisor 当天实盘建议（数据时效优先）
n_live = _train_and_save(trainer_live, MODEL_VERSION_LIVE, x_filter_end=None)

print(f'\n汇总：v1 落库 {n_live} 个；v1_oos 落库 {n_oos} 个')

  开始训练 [v1_oos]   截断到 20241231（OOS 模型）  cv.train_window=600

=== 000833.SZ 粤桂股份      样本 905 | y_5d_valid 905 | 特征=60维(完整) ===


18:17:55 | INFO    | [000833.SZ] h=3d: avg_ic=+0.0000 avg_rmse=0.0664 hit=52.17% final_n=600 iter=50 params=default
18:18:06 | INFO    | [000833.SZ] h=5d: avg_ic=+0.0360 avg_rmse=0.0934 hit=54.53% final_n=600 iter=50 params=default
18:18:23 | INFO    | [000833.SZ] h=10d: avg_ic=+0.2219 avg_rmse=0.1428 hit=58.46% final_n=600 iter=50 params=default



=== 002156.SZ 通富微电      样本 969 | y_5d_valid 969 | 特征=60维(完整) ===


18:18:36 | INFO    | [002156.SZ] h=3d: avg_ic=+0.0000 avg_rmse=0.0609 hit=50.04% final_n=600 iter=50 params=default
18:18:54 | INFO    | [002156.SZ] h=5d: avg_ic=+0.0585 avg_rmse=0.0775 hit=50.92% final_n=600 iter=50 params=default
18:19:19 | INFO    | [002156.SZ] h=10d: avg_ic=+0.2317 avg_rmse=0.1023 hit=56.71% final_n=600 iter=50 params=default



=== 002594.SZ 比亚迪       样本 905 | y_5d_valid 905 | 特征=60维(完整) ===


...


=== 600118.SH 中国卫星      样本 1296 | y_5d_valid 1289 | 特征=60维(完整) ===


18:31:35 | INFO    | [600118.SH] h=3d: avg_ic=+0.0000 avg_rmse=0.0569 hit=52.04% final_n=1000 iter=50 params=default
18:31:51 | INFO    | [600118.SH] h=5d: avg_ic=+0.1007 avg_rmse=0.0761 hit=50.84% final_n=1000 iter=50 params=default
18:32:13 | INFO    | [600118.SH] h=10d: avg_ic=+0.1052 avg_rmse=0.1078 hit=57.11% final_n=1000 iter=50 params=default



=== 516120.SH 化工50ETF   样本 1255 | y_5d_valid 1248 | 特征=20维(ETF精简) ===


18:32:47 | INFO    | [516120.SH] h=3d: avg_ic=+0.1345 avg_rmse=0.0268 hit=48.04% final_n=1000 iter=50 params=ETF
18:33:31 | INFO    | [516120.SH] h=5d: avg_ic=+0.1764 avg_rmse=0.0346 hit=55.34% final_n=1000 iter=108 params=ETF
18:33:58 | INFO    | [516120.SH] h=10d: avg_ic=+0.2096 avg_rmse=0.0474 hit=44.66% final_n=1000 iter=51 params=ETF



→ 落库 21 个 (code, horizon) 模型 [version=v1]

汇总：v1 落库 21 个；v1_oos 落库 21 个


## 5. 评估报告

汇总每票每 horizon 的 walk-forward CV 平均 IC / RMSE / 方向命中率。

In [5]:
from invest_model.ml import list_registry

show_cols = ['code', 'horizon', 'n_samples', 'n_features',
             'cv_avg_ic', 'cv_avg_rmse', 'cv_hit_rate', 'train_start', 'train_end']

for version_label, version in [('live (v1)', MODEL_VERSION_LIVE), ('OOS (v1_oos)', MODEL_VERSION_OOS)]:
    print('=' * 70)
    print(f'  {version_label}')
    print('=' * 70)
    registry = list_registry(engine, codes=all_codes, version=version)
    if registry.empty:
        print(f'  无注册模型 (version={version})')
        continue
    summary = registry[show_cols].copy()
    summary['name'] = summary['code'].map(code_name_map)
    summary = summary[['code', 'name'] + show_cols[1:]]
    display(summary.sort_values(['code', 'horizon']).reset_index(drop=True))

    avg_ic_5d = registry[registry['horizon'] == 5]['cv_avg_ic'].mean()
    avg_hit_5d = registry[registry['horizon'] == 5]['cv_hit_rate'].mean()
    print(f'  → horizon=5d 全标的均值: avg_ic={avg_ic_5d:+.4f}  avg_hit={avg_hit_5d:.2%}')
    print()

  live (v1)


,code,name,horizon,n_samples,n_features,cv_avg_ic,cv_avg_rmse,cv_hit_rate,train_start,train_end
0,000833.SZ,粤桂股份,3,1000,60,0.06129,0.05733,0.5431,20220304,20260420
1,000833.SZ,粤桂股份,5,1000,60,-0.03493,0.07236,0.6261,20220302,20260416
2,000833.SZ,粤桂股份,10,1000,60,0.10015,0.09668,0.6299,20220223,20260409
3,002156.SZ,通富微电,3,1000,60,0.11231,0.05144,0.5130,20220304,20260420
4,002156.SZ,通富微电,5,1000,60,0.28155,0.06430,0.6528,20220302,20260416
5,002156.SZ,通富微电,10,1000,60,0.33083,0.08407,0.7348,20220223,20260409
6,002594.SZ,比亚迪,3,1000,60,0.02481,0.07772,0.4979,20220304,20260420
7,002594.SZ,比亚迪,5,1000,60,0.18544,0.09913,0.4534,20220302,20260416
8,002594.SZ,比亚迪,10,1000,60,-0.15267,0.16537,0.4788,20220223,20260409
9,002648.SZ,卫星化学,3,1000,60,0.05734,0.04517,0.4697,20220304,20260420


  → horizon=5d 全标的均值: avg_ic=+0.1500  avg_hit=55.28%

  OOS (v1_oos)


,code,name,horizon,n_samples,n_features,cv_avg_ic,cv_avg_rmse,cv_hit_rate,train_start,train_end
0,000833.SZ,粤桂股份,3,600,60,0.00000,0.06643,0.5217,20220630,20241217
1,000833.SZ,粤桂股份,5,600,60,0.03599,0.09335,0.5453,20220630,20241217
2,000833.SZ,粤桂股份,10,600,60,0.22192,0.14276,0.5846,20220630,20241217
3,002156.SZ,通富微电,3,600,60,0.00000,0.06091,0.5004,20220630,20241217
4,002156.SZ,通富微电,5,600,60,0.05850,0.07752,0.5092,20220630,20241217
5,002156.SZ,通富微电,10,600,60,0.23168,0.10233,0.5671,20220630,20241217
6,002594.SZ,比亚迪,3,600,60,0.00000,0.03620,0.4068,20220630,20241217
7,002594.SZ,比亚迪,5,600,60,0.00000,0.04812,0.3861,20220630,20241217
8,002594.SZ,比亚迪,10,600,60,0.14422,0.06641,0.4758,20220630,20241217
9,002648.SZ,卫星化学,3,600,60,0.05312,0.03397,0.5036,20220630,20241217


  → horizon=5d 全标的均值: avg_ic=+0.0496  avg_hit=53.17%



### 5.1 特征重要性（合并所有票/horizon）

In [6]:
from invest_model.ml import load_artifact

import_rows = []
for code in all_codes:
    for h in LABEL_HORIZONS:
        art = load_artifact(engine, code, h, version=MODEL_VERSION_LIVE)
        if art is None or art.model is None:
            continue
        booster = art.model.get_booster()
        scores = booster.get_score(importance_type='gain')
        for fname, gain in scores.items():
            import_rows.append({
                'code': code,
                'horizon': h,
                'feature': fname,
                'gain': float(gain),
            })

if import_rows:
    imp_df = pd.DataFrame(import_rows)
    top_features = (
        imp_df.groupby('feature')['gain'].sum()
        .sort_values(ascending=False).head(20)
    )
    display(top_features.to_frame('total_gain'))
else:
    print('无特征重要性数据')

,total_gain
feature,
rsi_extreme,2.548498
pb_rank,2.234746
tech_macd_hist,2.112072
trig_pos_in_20d,1.873343
tech_volatility_20,1.865235
pe_rank,1.784647
comp_lag1,1.754229
trend_ma_slope,1.749383
comp_roll_max_10d,1.733891
